In [1]:
import numpy as np
from environments.simple_trading_env import SimpleTradingEnv
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import pandas as pd
import torch
import matplotlib.pyplot as plt

# === CONFIGURATION ===
DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_PATH = "trading_bot_hybrid"
LOOKBACK_WINDOW = 288

# Load data
df = pd.read_pickle(DATA_PATH)
print(f'✓ Loaded {len(df):,} rows for {DATA_SYMBOL} {DATA_TIMEFRAME}')

# Test on unseen data
total_timesteps = 1000
test_start = 215_536
test_data = df.iloc[test_start:test_start + total_timesteps].reset_index(drop=True)
print(f'✓ Testing on rows {test_start:,} to {test_start + total_timesteps:,}')

# Create test environment
test_env = SimpleTradingEnv(test_data, device="cuda", lookback_window=LOOKBACK_WINDOW)
test_env = Monitor(test_env)
test_env = DummyVecEnv([lambda: test_env])

# Load trained model
model = PPO.load(MODEL_PATH, env=test_env, device="cuda")
print(f'✓ Loaded model from {MODEL_PATH}\n')

# === FEATURE ACTIVATION TRACKING ===
extractor = model.policy.features_extractor
HOOKABLE_PATTERNS = ['_cnn', '_output', '_encoder', '_transformer', '_mlp']

# Auto-discover all hookable modules
available_features = {}
for attr_name in dir(extractor):
    if attr_name.startswith('_'):
        continue
    if any(pattern in attr_name for pattern in HOOKABLE_PATTERNS):
        attr = getattr(extractor, attr_name)
        if isinstance(attr, torch.nn.Module):
            display_name = attr_name.replace('_', ' ').title().replace(' ', '_')
            available_features[attr_name] = display_name

✓ Loaded 264,323 rows for BTCUSDT 5m
✓ Testing on rows 215,536 to 216,536
Info: Dropped 99 rows due to NaNs after adding indicators.
✓ Loaded model from trading_bot_hybrid

✓ Loaded model from trading_bot_hybrid



In [ ]:
from copy import deepcopy

print(f"\n🔍 Available hookable modules ({len(available_features)}):")
for module_name, display_name in available_features.items():
    print(f"   - {display_name}")

# Setup hooks to capture activations
activations = {name: [] for name in available_features.keys()}

def make_hook(feature_name):
    def hook(module, input, output):
        # Capture mean absolute activation
        if isinstance(output, torch.Tensor):
            activations[feature_name].append(output.abs().mean().item())
    return hook

# Register hooks
hooks = []
for attr_name in available_features.keys():
    module = getattr(extractor, attr_name)
    hook = module.register_forward_hook(make_hook(attr_name))
    hooks.append(hook)

print("\n" + "="*80)
print("Running evaluation...")
print("="*80)

# Run evaluation
obs = test_env.reset()
done = False
total_reward = 0
episode_rewards = []
step_count = 0
last_env = None

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = test_env.step(action)
    total_reward += reward[0]
    episode_rewards.append(reward[0])
    step_count += 1

    if test_env.envs[0].env.data_len == info[0].get('step') + 3:
        last_env = deepcopy(test_env.envs[0].env)
    # Store environment history for trade analysis
    if done[0]:
        print("Episode finished.")
# Remove hooks
for hook in hooks:
    hook.remove()

# Print results
print("\n" + "="*80)
print("EVALUATION REPORT")
print("="*80)

print("\n📊 REWARD STATISTICS:")
print(f"   Total Reward:       {total_reward:+8.2f}")
print(f"   Min Reward:         {min(episode_rewards):+8.2f}")
print(f"   Max Reward:         {max(episode_rewards):+8.2f}")
print(f"   Avg Reward/Step:  {total_reward/step_count:+8.4f}")
print(f"   Total Steps:         {step_count:>6}")

# Feature activations
print(f"\n📈 FEATURE ACTIVATIONS ({len(available_features)} groups):")
feature_stats = {name: np.mean(acts) if acts else 0.0 for name, acts in activations.items()}
sorted_features = sorted(feature_stats.items(), key=lambda x: x[1], reverse=True)

print(f"\n  Top 10 Most Active Features:")
for i, (name, avg_activation) in enumerate(sorted_features[:10], 1):
    display_name = available_features[name]
    print(f"    {i:2d}. {display_name:30s} : {avg_activation:.4f}")


🔍 Available hookable modules (6):
   - Account_Encoder
   - Macro_Cnn
   - Meso_Cnn
   - Micro_Spatial_Mlp
   - Micro_Temporal_Cnn
   - Position_Encoder

Running evaluation...
Episode finished.

EVALUATION REPORT

📊 REWARD STATISTICS:
   Total Reward:         +25.68
   Min Reward:            -1.50
   Max Reward:            +0.90
   Avg Reward/Step:   +0.0420
   Total Steps:            612

📈 FEATURE ACTIVATIONS (6 groups):

  Top 10 Most Active Features:
     1. Position_Encoder               : 0.2219
     2. Meso_Cnn                       : 0.2034
     3. Micro_Spatial_Mlp              : 0.1794
     4. Macro_Cnn                      : 0.1687
     5. Account_Encoder                : 0.1209
     6. Micro_Temporal_Cnn             : 0.0960
Episode finished.

EVALUATION REPORT

📊 REWARD STATISTICS:
   Total Reward:         +25.68
   Min Reward:            -1.50
   Max Reward:            +0.90
   Avg Reward/Step:   +0.0420
   Total Steps:            612

📈 FEATURE ACTIVATIONS (6 groups):



In [3]:
# === COMPREHENSIVE TRADING REPORT ===
print("\n" + "="*80)
print("TRADING PERFORMANCE REPORT")
print("="*80)

action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}

# Count actions from history
action_counts = {0: 0, 1: 0, 2: 0, 3: 0}
position_states = []

for env_state in last_env.history:
    action = env_state.get('action', [0])
    if isinstance(action, (list, np.ndarray)):
        action_id = int(action[0])
    else:
        action_id = int(action)
    action_counts[action_id] = action_counts.get(action_id, 0) + 1
    
    position_size = env_state.get('position_size', 0)
    position_states.append(position_size)

total_actions = sum(action_counts.values())

# === 1. ACTION DISTRIBUTION ===
print("\n📊 ACTION DISTRIBUTION:")
for action_id, count in sorted(action_counts.items()):
    pct = (count / total_actions * 100) if total_actions > 0 else 0
    bar = '█' * int(pct / 2)  # Visual bar
    print(f"   {action_names[action_id]:6s}: {count:5d} ({pct:5.1f}%) {bar}")

# === 2. POSITION DISTRIBUTION ===
flat_steps = sum(1 for ps in position_states if ps == 0)
long_steps = sum(1 for ps in position_states if ps > 0)
short_steps = sum(1 for ps in position_states if ps < 0)
total_steps = len(position_states)

print("\n📈 POSITION DISTRIBUTION:")
print(f"   FLAT : {flat_steps:5d} steps ({flat_steps/total_steps*100:5.1f}%)")
print(f"   LONG : {long_steps:5d} steps ({long_steps/total_steps*100:5.1f}%)")
print(f"   SHORT: {short_steps:5d} steps ({short_steps/total_steps*100:5.1f}%)")

# === 3. BALANCE PERFORMANCE ===
initial_balance = last_env.initial_balance
final_balance = last_env.broker.current_balance
balance_change = final_balance - initial_balance
balance_change_pct = (balance_change / initial_balance) * 100

print("\n💰 BALANCE PERFORMANCE:")
print(f"   Initial: ${initial_balance:>10,.2f}")
print(f"   Final:   ${final_balance:>10,.2f}")
print(f"   Change:  ${balance_change:>+10,.2f} ({balance_change_pct:+.2f}%)")

# === 4. COMPLETED TRADES ===
completed_trades = last_env.broker.trade_history
closed_trades = [t for t in completed_trades if t.get('status') == 'CLOSED']

if len(closed_trades) > 0:
    total_pnl = sum(t.get('pnl', 0) for t in closed_trades)
    wins = [t for t in closed_trades if t.get('pnl', 0) > 0]
    losses = [t for t in closed_trades if t.get('pnl', 0) <= 0]
    
    print(f"\n📋 TRADE SUMMARY:")
    print(f"   Total Trades:  {len(closed_trades):>5}")
    print(f"   Wins:          {len(wins):>5} ({len(wins)/len(closed_trades)*100:5.1f}%)")
    print(f"   Losses:        {len(losses):>5} ({len(losses)/len(closed_trades)*100:5.1f}%)")
    print(f"   Total PnL:     ${total_pnl:>+10,.2f}")
    print(f"   Avg PnL:       ${total_pnl/len(closed_trades):>+10,.2f}")
    if wins:
        print(f"   Avg Win:       ${sum(t['pnl'] for t in wins)/len(wins):>+10,.2f}")
    if losses:
        print(f"   Avg Loss:      ${sum(t['pnl'] for t in losses)/len(losses):>+10,.2f}")
    
    # Exit reason breakdown
    exit_reasons = {}
    for t in closed_trades:
        reason = t.get('reason', 'Unknown')
        exit_reasons[reason] = exit_reasons.get(reason, 0) + 1
    
    print(f"\n📊 EXIT REASONS:")
    for reason, count in sorted(exit_reasons.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(closed_trades)) * 100
        print(f"   {reason:20s}: {count:3d} ({pct:5.1f}%)")
    
    # === 5. DETAILED TRADE TABLE ===
    print("\n" + "="*80)
    print("DETAILED TRADE HISTORY")
    print("="*80)
    
    # Build step to action mapping
    step_to_action = {}
    for env_state in last_env.history:
        step = env_state.get('step', 0)
        action = env_state.get('action', [0])
        if isinstance(action, (list, np.ndarray)):
            action_id = int(action[0])
        else:
            action_id = int(action)
        step_to_action[step] = action_id
    
    # Create trades DataFrame
    trades_list = []
    for i, t in enumerate(closed_trades, 1):
        step_open = t.get('step_open', 0)
        action_id = step_to_action.get(step_open, 0)
        
        trades_list.append({
            '#': i,
            'Step': step_open,
            'Dir': action_names.get(action_id, '?'),
            'Entry': t.get('entry_price', 0),
            'Exit': t.get('exit_price', 0),
            'Duration': t.get('duration', 0),
            'PnL': t.get('pnl', 0),
            'PnL%': t.get('pnl_percent', 0) * 100,
            'Reason': t.get('reason', 'N/A'),
        })
    
    trades_df = pd.DataFrame(trades_list)
    
    # Apply color styling: green for profit, red for loss with black text
    def color_pnl(val):
        if val > 0:
            return 'background-color: #90EE90; color: black'  # Light green with black text
        elif val < 0:
            return 'background-color: #FF6B6B; color: black'  # Bright red with black text
        else:
            return 'color: black'
    
    # Display with styling (using map instead of deprecated applymap)
    styled_df = trades_df.style.format({
        'Entry': '${:,.2f}',
        'Exit': '${:,.2f}',
        'PnL': '${:+,.2f}',
        'PnL%': '{:+.2f}%',
        'Duration': '{:.0f}',
    }).map(color_pnl, subset=['PnL'])
    
    display(styled_df)

else:
    print(f"\n⚠️  NO COMPLETED TRADES")

# === 6. OPEN POSITION (if any) ===
final_state = last_env.history[-1]
final_position = final_state.get('position_size', 0)

if final_position != 0:
    direction = 'LONG' if final_position > 0 else 'SHORT'
    unrealized_pnl = final_state.get('unrealized_pnl', 0)
    entry_price = final_state.get('entry_price', 0)
    current_price = final_state.get('current_price', 0)
    sl_price = final_state.get('stop_loss_price', 0)
    tp_price = final_state.get('take_profit_price', 0)
    
    print(f"\n⚠️  OPEN POSITION:")
    print(f"   Direction:      {direction}")
    print(f"   Entry Price:    ${entry_price:,.2f}")
    print(f"   Current Price:  ${current_price:,.2f}")
    print(f"   Stop Loss:      ${sl_price:,.2f}")
    print(f"   Take Profit:    ${tp_price:,.2f}")
    print(f"   Unrealized PnL: ${unrealized_pnl:+,.2f}")

print("\n" + "="*80)


TRADING PERFORMANCE REPORT

📊 ACTION DISTRIBUTION:
   HOLD  :   304 ( 33.8%) ████████████████
   LONG  :   258 ( 28.7%) ██████████████
   SHORT :    40 (  4.4%) ██
   CLOSE :   297 ( 33.0%) ████████████████

📈 POSITION DISTRIBUTION:
   FLAT :   585 steps ( 65.1%)
   LONG :   274 steps ( 30.5%)
   SHORT:    40 steps (  4.4%)

💰 BALANCE PERFORMANCE:
   Initial: $ 10,000.00
   Final:   $  5,384.21
   Change:  $ -4,615.79 (-46.16%)

📋 TRADE SUMMARY:
   Total Trades:    297
   Wins:            115 ( 38.7%)
   Losses:          182 ( 61.3%)
   Total PnL:     $ -2,777.33
   Avg PnL:       $     -9.35
   Avg Win:       $    +17.76
   Avg Loss:      $    -26.48

📊 EXIT REASONS:
   Manual Close        : 287 ( 96.6%)
   SL                  :  10 (  3.4%)

DETAILED TRADE HISTORY


,#,Step,Dir,Entry,Exit,Duration,PnL,PnL%,Reason
0,1,288,LONG,"$103,943.13","$103,725.85",3,$-54.98,-55.01%,Manual Close
1,2,292,SHORT,"$103,829.97","$103,837.87",1,$-6.59,-6.64%,Manual Close
2,3,294,LONG,"$103,879.99","$103,835.84",3,$-15.32,-15.44%,Manual Close
3,4,298,LONG,"$103,819.85","$104,057.92",2,$+54.97,+55.50%,Manual Close
4,5,301,LONG,"$104,207.58","$104,148.38",1,$-21.29,-21.39%,Manual Close
5,6,303,SHORT,"$104,268.17","$104,309.99",1,$-16.92,-17.05%,Manual Close
6,7,305,LONG,"$104,291.70","$104,299.99",2,$-3.43,-3.47%,Manual Close
7,8,308,SHORT,"$104,327.98","$104,349.20",1,$-11.79,-11.91%,Manual Close
8,9,310,SHORT,"$104,251.37","$104,197.91",1,$+9.33,+9.45%,Manual Close
9,10,312,SHORT,"$104,130.07","$104,253.96",1,$-42.70,-43.22%,Manual Close



⚠️  OPEN POSITION:
   Direction:      LONG
   Entry Price:    $102,974.66
   Current Price:  $102,974.66
   Stop Loss:      $102,517.17
   Take Profit:    $104,118.39
   Unrealized PnL: $+0.00

